In [22]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv('../data/raw/AirBnB_BandungRaya_All.csv')

# Pembersihan harga & rating 
df['harga'] = df['harga'].str.replace('Rp', '', regex=False).str.replace(',', '', regex=False).str.strip().astype(int)
df['rating'] = pd.to_numeric(df['rating'].replace('New', np.nan), errors='coerce')

df.head(2)

,nama,harga,rating,jumlah_review,link,wilayah
0,Home in Cibeunying Kaler - Serene Pet-Friendly...,9848899,5.00,33,https://www.airbnb.com/rooms/1647712818622104523,Kota Bandung
1,"Apartment in Cidadap - Dago, Ciumbuleuit, ITB ...",3446781,4.98,132,https://www.airbnb.com/rooms/1583745571091269827,Kota Bandung


In [23]:
def extract_tipe(nama):
    # Mengambil teks sebelum kata ' in '
    match = re.search(r'^(.*?)\s+in\s+', str(nama), re.IGNORECASE)
    return match.group(1).strip() if match else 'Other'

def extract_lokasi(nama):
    # Mengambil teks di antara ' in ' dan ' - '
    match = re.search(r'\s+in\s+(.*?)\s+-', str(nama), re.IGNORECASE)
    return match.group(1).strip() if match else 'Unknown'

df['tipe_raw'] = df['nama'].apply(extract_tipe)
df['lokasi_raw'] = df['nama'].apply(extract_lokasi)

print("Jumlah Tipe Properti:")
print(df['tipe_raw'].value_counts().head(10))

print("\nJumlah Lokasi (Kecamatan/Area) Terbanyak:")
print(df['lokasi_raw'].value_counts().head(10))

Jumlah Tipe Properti:
tipe_raw
Home           250
Villa          127
Apartment      101
Room            19
Guesthouse      11
Condo           10
Cabin           10
Rooms            9
Tiny home        7
Guest suite      5
Name: count, dtype: int64

Jumlah Lokasi (Kecamatan/Area) Terbanyak:
lokasi_raw
Megamendung          68
Cisarua              51
Parompong            35
Lembang              24
Cimenyan             23
Cicendo              21
Bogor Selatan        21
Cidadap              16
Kecamatan Cidadap    16
Babakan Madang       15
Name: count, dtype: int64


In [24]:
# Ambil semua lokasi unik
lokasi_unik = df['lokasi_raw'].value_counts().reset_index()
lokasi_unik.columns = ['lokasi_raw', 'jumlah']

print(f"Total lokasi unik: {len(lokasi_unik)}")
lokasi_unik.to_csv('../data/reference/daftar_lokasi_mentah.csv', index=False)

Total lokasi unik: 82


In [25]:
print(df.columns.tolist())
mapping = pd.read_csv('../data/reference/kecamatan_mapping.csv')
print(mapping.columns.tolist())

['nama', 'harga', 'rating', 'jumlah_review', 'link', 'wilayah', 'tipe_raw', 'lokasi_raw']
['lokasi_raw', '\tkecamatan', '\tkabupaten_kota', '\tregion']


In [26]:
# Load mapping
mapping = pd.read_csv('../data/reference/kecamatan_mapping.csv')

# Validasi mapping
missing_lokasi = set(df['lokasi_raw']) - set(mapping['lokasi_raw'])
if missing_lokasi:
    print(f"Ada sebanyak {len(missing_lokasi)} lokasi yang tidak ada di mapping:")
    for lokasi in missing_lokasi:
        print(f"- {lokasi}")

# Cek missing
missing_lokasi = set(df['lokasi_raw']) - set(mapping['lokasi_raw'])
if missing_lokasi:
    print(f"Ada sebanyak {len(missing_lokasi)} lokasi yang tidak ada di mapping:")
    for lokasi in missing_lokasi:
        print(f"- {lokasi}")

# Merge mapping ke df
df = df.merge(mapping, on='lokasi_raw', how='left')
print(df.head())

                                                nama     harga  rating  \
0  Home in Cibeunying Kaler - Serene Pet-Friendly...   9848899    5.00   
1  Apartment in Cidadap - Dago, Ciumbuleuit, ITB ...   3446781    4.98   
2         Home in Sukajadi - Pine Breeze W/ Backyard  12500000    4.95   
3  Apartment in Cidadap - Abony | Bandung | PVJ |...   2054119    4.87   
4                     Home in Cidadap - Wangiterrace   9883877    4.98   

   jumlah_review                                              link  \
0             33  https://www.airbnb.com/rooms/1647712818622104523   
1            132  https://www.airbnb.com/rooms/1583745571091269827   
2            112  https://www.airbnb.com/rooms/1400529902543354123   
3             38  https://www.airbnb.com/rooms/1496783488914922824   
4             51  https://www.airbnb.com/rooms/1504314537783385401   

        wilayah   tipe_raw        lokasi_raw         \tkecamatan  \
0  Kota Bandung       Home  Cibeunying Kaler  \tCibeunying Kaler  

In [27]:
# Bersihkan karakter tab (\t) di kolom kecamatan, kabupaten, dan region
df.columns = df.columns.str.replace('\t','').str.strip()  

# Cek apakah ada gagal merge
missing_region = df[df['region'].isna()]
print(f"Jumlah baris tanpa region: {len(missing_region)}")

# Cek kolom setelah merge
print(df.columns.tolist())

Jumlah baris tanpa region: 0
['nama', 'harga', 'rating', 'jumlah_review', 'link', 'wilayah', 'tipe_raw', 'lokasi_raw', 'kecamatan', 'kabupaten_kota', 'region']


## Ringkasan Progress Cleaning (WIP)

- **Ekstraksi Tipe & Lokasi**: Berhasil mengekstrak `tipe_raw` (Home, Villa, Apartment, dll) dan `lokasi_raw` dari kolom `nama` menggunakan regex.
- **Mapping Kecamatan**: Berhasil menggabungkan data dengan `kecamatan_mapping.csv` untuk mendapatkan kolom `kecamatan` dan `region` (Bandung Raya vs Bogor). Karakter `\t` pada nama kolom sudah dibersihkan.
- **Keputusan Tertunda (Next Steps)**: 
  - Mengevaluasi hasil *sanity check* (crosstab) antara `tipe_raw` dan fasilitas. Jika persentasenya tidak masuk akal (bias), fitur fasilitas akan dibuang dan hanya menggunakan `bedrooms` serta `tipe_raw`.
  - Melakukan transformasi logaritma (`log_harga`) dan perhitungan Bayesian rating (`rating_adj`).
  - Menyimpan dataset final ke `data/processed/airbnb_clean.csv`.

In [28]:
import re
import numpy as np

# Ekstrak jumlah kamar dari kolom 'nama'
def extract_bedrooms(text):
    text = str(text)
    # Mencari angka yang diikuti oleh BR, Bedroom, Kamar, atau Bed (contoh: 3BR, 3 Bedroom, 2 Kamar)
    match = re.search(r'(\d+)\s*(?:BR|Bedroom|Kamar|Beds?)', text, re.IGNORECASE)
    return int(match.group(1)) if match else np.nan

# Terapkan ke dataframe
df['bedrooms'] = df['nama'].apply(extract_bedrooms)

# Pengecekan: Berapa banyak baris yang berhasil
total_baris = len(df)
berhasil = df['bedrooms'].notna().sum()
gagal = df['bedrooms'].isna().sum()

print(f"Total baris: {total_baris}")
print(f"Berhasil ekstrak kamar: {berhasil} ({(berhasil/total_baris)*100:.1f}%)")
print(f"Tidak ada info kamar (NaN): {gagal} ({(gagal/total_baris)*100:.1f}%)")

# 4. Pengecekan: Distribusi jumlah kamar
print("\nDistribusi jumlah kamar:")
print(df['bedrooms'].value_counts().sort_index())

# 5. Pengecekan: Contoh nama properti yang tidak ada info kamarnya
print("\nContoh nama properti yang tidak ada info kamarnya:")
contoh_gagal = df[df['bedrooms'].isna()]['nama'].head(10).tolist()
for nama in contoh_gagal:
    print(f"- {nama}")

Total baris: 573
Berhasil ekstrak kamar: 148 (25.8%)
Tidak ada info kamar (NaN): 425 (74.2%)

Distribusi jumlah kamar:
bedrooms
1.0     5
2.0    59
3.0    39
4.0    26
5.0    10
6.0     4
7.0     3
8.0     2
Name: count, dtype: int64

Contoh nama properti yang tidak ada info kamarnya:
- Home in Cibeunying Kaler - Serene Pet-Friendly Family Lux Villa, Aqua Ville
- Apartment in Cidadap - Dago, Ciumbuleuit, ITB | Cozy & Comfort | 4 Guests
- Home in Sukajadi - Pine Breeze W/ Backyard
- Apartment in Cidadap - Abony | Bandung | PVJ | Lembang | Pool
- Home in Cidadap - Wangiterrace
- Home in Cidadap - Villa Zenka by Kava Stay
- Home in Parompong - Big Family Villa with open space, Coney Ville
- Apartment in Cicendo - Raj Estate– Lux Apartement @ Gateway Pasteur
- Apartment in Kecamatan Cidadap - CozyGreenStudioWMBalconySmartTV WIIM LEGO
- Home in Kecamatan Bandung Wetan - Upstairs Tamanari


In [29]:
# Hapus kolom bedrooms
kolom_buang = ['bedrooms']
df = df.drop(columns=[col for col in kolom_buang if col in df.columns])

# Transformasi Logaritma untuk harga karena distribusinya sangat condong ke kanan (right-skewed)
df['log_harga'] = np.log1p(df['harga'])
df['log_jumlah_review'] = np.log1p(df['jumlah_review'])

# Bayesian Rating (menyeimbangkan rating dengan jumlah review)
R0 = df['rating'].mean()
C = 15
df['rating_adj'] = ((df['jumlah_review'] * df['rating'] + C * R0) / (df['jumlah_review'] + C))

# Cek struktur akhir
print("Kolom akhir yang siap dianalisis:")
print(df.columns.tolist())

print("\nContoh data final:")
display(df[['nama', 'harga', 'log_harga', 'rating', 'rating_adj', 'tipe_raw', 'kecamatan', 'region']].head())

# # Simpan ke folder processed
# df.to_csv('../data/processed/airbnb_clean.csv', index=False)
# print("\n✅ Data bersih berhasil disimpan di '../data/processed/airbnb_clean.csv'!")

Kolom akhir yang siap dianalisis:
['nama', 'harga', 'rating', 'jumlah_review', 'link', 'wilayah', 'tipe_raw', 'lokasi_raw', 'kecamatan', 'kabupaten_kota', 'region', 'log_harga', 'log_jumlah_review', 'rating_adj']

Contoh data final:


,nama,harga,log_harga,rating,rating_adj,tipe_raw,kecamatan,region
0,Home in Cibeunying Kaler - Serene Pet-Friendly...,9848899,16.102870,5.00,4.968317,Home,\tCibeunying Kaler,\tBandung Raya
1,"Apartment in Cidadap - Dago, Ciumbuleuit, ITB ...",3446781,15.052952,4.98,4.971695,Apartment,\tCidadap,\tBandung Raya
2,Home in Sukajadi - Pine Breeze W/ Backyard,12500000,16.341239,4.95,4.943931,Home,\tSukajadi,\tBandung Raya
3,Apartment in Cidadap - Abony | Bandung | PVJ |...,2054119,14.535358,4.87,4.878099,Apartment,\tCidadap,\tBandung Raya
4,Home in Cidadap - Wangiterrace,9883877,16.106416,4.98,4.961503,Home,\tCidadap,\tBandung Raya


In [30]:
# Simpan ke folder processed
df.to_csv('../data/processed/airbnb_clean.csv', index=False)
print("\n✅ Data bersih berhasil disimpan di '../data/processed/airbnb_clean.csv'!")


✅ Data bersih berhasil disimpan di '../data/processed/airbnb_clean.csv'!


## Fase 2: Ringkasan Data Cleaning & Feature Engineering

Pada tahap ini, data mentah telah dibersihkan dan ditransformasi agar siap untuk analisis lebih lanjut. Berikut adalah ringkasan keputusan dan metodologi yang diterapkan:

### 1. Ekstraksi Informasi & Normalisasi Lokasi
- **Tipe & Lokasi**: Kolom `tipe_raw` (Villa, Apartment, Home, dll) dan `lokasi_raw` berhasil diekstrak dari teks kolom `nama` menggunakan Regex.
- **Mapping Kecamatan**: `lokasi_raw` yang berantakan (misal: "Kecamatan Cidadap", "Sumurbandung") dinormalisasi menggunakan *reference table* (`kecamatan_mapping.csv`). Data kemudian dikelompokkan ke dalam `kecamatan` dan `region` (Bandung Raya vs Bogor).

### 2. Pembuangan Fitur (Feature Selection)
- **Fasilitas & Jumlah Kamar (`bedrooms`)**: Ekstraksi `bedrooms` hanya berhasil pada ~25% data. Karena sebaran data yang hilang tidak merata (Missing Not At Random), melakukan imputasi (seperti mengisi dengan median) berisiko menimbulkan bias yang fatal. Oleh karena itu, variabel `bedrooms` dan fasilitas spesifik (pool, netflix, dll) **dibuang dari analisis**. 
- **Fokus Analisis**: Analisis akan berfokus pada variabel yang datanya valid secara keseluruhan: `harga`, `rating`, `jumlah_review`, `tipe_raw`, `kecamatan`, dan `region`.

### 3. Transformasi Statistik (Data Transformation)
- **Logaritma Natural (`log1p`) pada Harga**: Distribusi harga properti sangat *right-skewed* (condong ke kanan) dengan nilai ekstrem hingga Rp 94 juta. Transformasi logaritma diterapkan untuk menormalkan distribusi, menstabilkan varians (mengurangi heteroskedastisitas), dan memungkinkan interpretasi persentase pada pemodelan regresi. Standar ini umum digunakan dalam model regresi hedonik properti.
- **Bayesian Average pada Rating (`rating_adj`)**: Rating mentah (misal 5.0 dari 2 review) tidak sebanding dengan 4.8 dari 300 review. Metode *Bayesian Average* (dengan konstanta `C=15`) diterapkan untuk melakukan *shrinkage*—menarik rating dari properti dengan sedikit ulasan ke arah rata-rata global. Ini memastikan properti yang mendominasi peringkat adalah mereka yang benar-benar terbukti kualitasnya (memiliki banyak ulasan).

### 4. Output
- Dataset bersih dan final disimpan di `../data/processed/airbnb_clean.csv` untuk digunakan pada tahap Eksplorasi Data (EDA) dan Pemodelan.